# 🎥 Wan 2.1 3D Camera Trajectory Worker (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ha-Lun/swarmkit/blob/main/notebooks/wan2_colab_worker.ipynb)

This notebook runs the **Wan 2.1 Text-to-Video** foundation model with camera trajectory guidance (`orbit_360`, `orbit_180`, `pan`, `tilt`, `zoom`, `spiral`) and exposes a public **Gradio `--share` REST API**.

### Architecture Overview:
- **Wan 2.1 Pipeline**: Generates 3D turntable videos with photorealistic camera motion.
- **Gradio + FastAPI**: Provides an interactive Web UI and HTTP endpoints (`POST /api/generate`, `GET /health`, `GET /api/download/{file}`).
- **n8n Integration**: Dispatches generation jobs, downloads videos, and slices zero-padded WebP frames into `public/frames/` for `CanvasScrubber.tsx`.

---

In [ ]:
from google.colab import drive
import os

print("Connecting Google Drive...")
drive.mount('/content/drive')

# Set Hugging Face cache directory to your 2TB Google Drive
os.environ["HF_HOME"] = "/content/drive/MyDrive/ai_models/huggingface"
os.makedirs("/content/drive/MyDrive/ai_models/huggingface", exist_ok=True)
os.makedirs("/content/drive/MyDrive/outputs/wan2", exist_ok=True)

print("✅ Google Drive mounted! Models and outputs will be permanently saved to your 2TB Drive.")

## 1. Verify GPU Runtime

Ensure your Colab runtime is set to **GPU** (Runtime → Change runtime type → T4 or A100/L4).

In [ ]:
!nvidia-smi

import torch
print("\n--- PyTorch CUDA Status ---")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM: {vram:.2f} GB")
else:
    print("⚠️ Warning: No GPU detected. Switch runtime to GPU under Runtime > Change runtime type.")

## 2. Install Required Dependencies

Installs diffusers (v0.33+ with WanPipeline), gradio, accelerate, fastapi, uvicorn, and video utilities.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate gradio fastapi uvicorn imageio imageio-ffmpeg opencv-python sentencepiece ftfy huggingface_hub

## 3. Clone Repository or Fetch Worker Script

Pulls `scripts/wan2_colab_worker.py` into your Colab session.

In [ ]:
import os
import sys
import subprocess
import urllib.request

os.makedirs("scripts", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

worker_path = "scripts/wan2_colab_worker.py"
extract_path = "scripts/extract-frames.sh"

# Clone repository or fetch worker script if not already present
if not os.path.exists(worker_path):
    print("🔄 Attempting to clone repository from GitHub...")
    clone_res = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Ha-Lun/swarmkit.git", "/tmp/swarmkit"],
        capture_output=True,
        text=True
    )
    if clone_res.returncode == 0 and os.path.exists("/tmp/swarmkit/scripts/wan2_colab_worker.py"):
        subprocess.run(["cp", "/tmp/swarmkit/scripts/wan2_colab_worker.py", "scripts/"], check=True)
        if os.path.exists("/tmp/swarmkit/scripts/extract-frames.sh"):
            subprocess.run(["cp", "/tmp/swarmkit/scripts/extract-frames.sh", "scripts/"], check=True)
        print("✅ Cloned Ha-Lun/swarmkit and installed worker scripts.")
    else:
        err_msg = clone_res.stderr.strip() if clone_res.stderr else f"exit code {clone_res.returncode}"
        print(f"⚠️ Git clone failed ({err_msg}).")
        
        # Fallback 1: Direct raw file download
        raw_worker_url = "https://raw.githubusercontent.com/Ha-Lun/swarmkit/main/scripts/wan2_colab_worker.py"
        raw_extract_url = "https://raw.githubusercontent.com/Ha-Lun/swarmkit/main/scripts/extract-frames.sh"
        print(f"🔄 Attempting direct download from raw GitHub: {raw_worker_url} ...")
        try:
            urllib.request.urlretrieve(raw_worker_url, worker_path)
            try:
                urllib.request.urlretrieve(raw_extract_url, extract_path)
                os.chmod(extract_path, 0o755)
            except Exception:
                pass
            print("✅ Successfully fetched wan2_colab_worker.py via direct download.")
        except Exception as dl_err:
            print(f"⚠️ Direct download failed: {dl_err}")

    # Fallback 2: Generate standalone in-notebook fallback worker script if still missing
    if not os.path.exists(worker_path):
        print("🛠️ Generating standalone fallback worker script at scripts/wan2_colab_worker.py...")
        fallback_lines = [
            '#!/usr/bin/env python3',
            '# Standalone Fallback Worker for Wan 2.1 3D Camera Trajectory Generation',
            'import os, sys, time, argparse, logging',
            'logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")',
            'logger = logging.getLogger("wan2-worker-fallback")',
            '',
            'CAMERA_PRESETS = {',
            '    "orbit_360": "360-degree orbit camera revolving smoothly around the subject, seamless circular turntable pan, 3D rotating view, centered subject, photorealistic depth",',
            '    "orbit_180": "180-degree semicircular camera orbit around the subject from front to profile view, smooth cinematic arc motion",',
            '    "pan_left": "smooth horizontal camera pan gliding from right to left across the subject, tracking shot",',
            '    "pan_right": "smooth horizontal camera pan gliding from left to right across the subject, tracking shot",',
            '    "tilt_up": "vertical camera tilt moving steadily upwards from low angle to high angle, cinematic sweep",',
            '    "tilt_down": "vertical camera tilt moving steadily downwards from high angle to low angle",',
            '    "zoom_in": "cinematic dolly in zoom moving steadily towards the subject, centered depth perspective",',
            '    "zoom_out": "cinematic dolly out zoom pulling back steadily away from the subject, wide perspective",',
            '    "spiral": "dynamic 3D spiral camera trajectory ascending while orbiting smoothly around the subject",',
            '    "custom": ""',
            '}',
            '',
            'def main():',
            '    parser = argparse.ArgumentParser(description="Wan 2.1 Colab Worker")',
            '    parser.add_argument("--model", type=str, default="Wan-AI/Wan2.1-T2V-1.3B-Diffusers")',
            '    parser.add_argument("--share", action="store_true")',
            '    parser.add_argument("--port", type=int, default=7860)',
            '    parser.add_argument("--output-dir", type=str, default="./outputs")',
            '    args = parser.parse_args()',
            '',
            '    os.makedirs(args.output_dir, exist_ok=True)',
            '    import gradio as gr',
            '',
            '    pipeline = None',
            '    try:',
            '        import torch',
            '        from diffusers import WanPipeline',
            '        logger.info(f"Loading WanPipeline from {args.model}...")',
            '        dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16',
            '        pipeline = WanPipeline.from_pretrained(args.model, torch_dtype=dtype)',
            '        if torch.cuda.is_available():',
            '            pipeline.enable_model_cpu_offload()',
            '            if hasattr(pipeline, "enable_vae_tiling"):',
            '                pipeline.enable_vae_tiling()',
            '        logger.info("WanPipeline loaded successfully.")',
            '    except Exception as e:',
            '        logger.warning(f"Could not load WanPipeline ({e}). Procedural generator will be used if needed.")',
            '',
            '    def generate(prompt, camera_motion="orbit_360", frames=81, fps=16):',
            '        suffix = CAMERA_PRESETS.get(camera_motion, "")',
            '        full_prompt = f"{prompt.strip()}, {suffix}".strip(", ")',
            '        logger.info(f"Generating: {full_prompt}")',
            '        ts = int(time.time())',
            '        out_path = os.path.join(args.output_dir, f"wan2_{camera_motion}_{ts}.mp4")',
            '',
            '        if pipeline is not None:',
            '            try:',
            '                res = pipeline(prompt=full_prompt, num_frames=int(frames)).frames[0]',
            '                import imageio',
            '                imageio.mimsave(out_path, res, fps=int(fps))',
            '                return out_path, {"status": "success", "output": out_path, "prompt": full_prompt}',
            '            except Exception as gen_err:',
            '                logger.error(f"Inference error: {gen_err}")',
            '',
            '        try:',
            '            import cv2, numpy as np',
            '            fourcc = cv2.VideoWriter_fourcc(*"mp4v")',
            '            vw = cv2.VideoWriter(out_path, fourcc, float(fps), (640, 360))',
            '            for i in range(int(frames)):',
            '                frame = np.zeros((360, 640, 3), dtype=np.uint8)',
            '                angle = (i / int(frames)) * 2 * 3.14159',
            '                cx = int(320 + 100 * np.cos(angle))',
            '                cy = int(180 + 50 * np.sin(angle))',
            '                cv2.circle(frame, (cx, cy), 30, (0, 200, 255), -1)',
            '                cv2.putText(frame, f"Wan2 {camera_motion} ({i+1}/{frames})", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)',
            '                vw.write(frame)',
            '            vw.release()',
            '        except Exception:',
            '            with open(out_path, "wb") as f:',
            '                f.write(b"MOCK_MP4")',
            '        return out_path, {"status": "mock_generated", "output": out_path, "prompt": full_prompt}',
            '',
            '    with gr.Blocks(title="Wan 2.1 3D Camera Trajectory Worker") as demo:',
            '        gr.Markdown("# 🎥 Wan 2.1 3D Camera Trajectory Worker (Fallback Mode)")',
            '        with gr.Row():',
            '            p_in = gr.Textbox(label="Prompt", value="cyberpunk flying car, sleek metallic, glowing neon trim", lines=2)',
            '            c_in = gr.Dropdown(choices=list(CAMERA_PRESETS.keys()), value="orbit_360", label="Camera Preset")',
            '        btn = gr.Button("Generate Video", variant="primary")',
            '        v_out = gr.Video(label="Generated Trajectory Video")',
            '        meta_out = gr.JSON(label="Metadata")',
            '        btn.click(fn=generate, inputs=[p_in, c_in], outputs=[v_out, meta_out])',
            '',
            '        @demo.app.get("/health")',
            '        def health():',
            '            return {"status": "ok", "worker": "wan2-colab-fallback"}',
            '',
            '    demo.launch(server_name="0.0.0.0", server_port=args.port, share=args.share)',
            '',
            'if __name__ == "__main__":',
            '    main()',
            ''
        ]
        with open(worker_path, "w", encoding="utf-8") as f:
            f.write("\n".join(fallback_lines))
        
        if not os.path.exists(extract_path):
            with open(extract_path, "w", encoding="utf-8") as ef:
                ef.write('#!/usr/bin/env bash\nffmpeg -y -i "$1" -vf "fps=${3:-30},scale=${4:-832}:${5:-480}" -c:v libwebp -lossless 0 -q:v 85 -compression_level 4 "${2:-frames}/frame_%04d.webp"\n')
            os.chmod(extract_path, 0o755)

        print("✅ Standalone fallback worker script created at scripts/wan2_colab_worker.py.")
        print("💡 Clean Prompt / Next Steps:")
        print("   1. If Ha-Lun/swarmkit is private, clone with a personal access token:")
        print("      !git clone https://<GITHUB_TOKEN>@github.com/Ha-Lun/swarmkit.git /tmp/swarmkit")
        print("   2. Or upload scripts/wan2_colab_worker.py directly to Colab under 'scripts/'")
        print("   3. Otherwise, the generated fallback script is ready to run in Step 4.")

if os.path.exists(worker_path):
    print("\n✅ Worker script ready:")
    !ls -l scripts/wan2_colab_worker.py
else:
    print("\n❌ Error: Worker script could not be prepared. Please upload wan2_colab_worker.py into 'scripts/'.")

## 4. Launch Worker Server with Public Gradio Link

This cell starts the Gradio app with `--share`. Gradio will output a public URL in the format:
`Running on public URL: https://xxxxxxxx.gradio.live`

> **Model Options:**
> - `Wan-AI/Wan2.1-T2V-1.3B-Diffusers` (Recommended for free T4 GPU, ~12GB VRAM with CPU offload)
> - `Wan-AI/Wan2.1-T2V-14B-Diffusers` (For Colab Pro A100 with 40GB+ VRAM)

In [ ]:
MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

# Launch server in background/foreground
!python scripts/wan2_colab_worker.py \
    --model {MODEL_ID} \
    --share \
    --port 7860 \
    --output-dir ./outputs

## 5. Next Steps: Triggering Generation

Once the server is running, copy the **`https://xxxxxxxx.gradio.live`** public link:

1. **Trigger via Swarm Script**:
   ```bash
   ./scripts/trigger-3d-animation.sh \
     --prompt "cyberpunk flying car, sleek metallic, glowing neon trim" \
     --camera orbit_360 \
     --colab-url "https://xxxxxxxx.gradio.live"
   ```

2. **Trigger via n8n Webhook**:
   Set `WAN2_COLAB_URL=https://xxxxxxxx.gradio.live` in n8n environment variables, then POST to `/webhook/generate-3d-animation`.

3. **Inspect in Web UI**:
   Open `https://xxxxxxxx.gradio.live` in your browser to test generation and camera angles interactively.